# C-to-Rust Migration with Qwen2.5-Coder-7B + Online GRPO Training

Translates a C repository to Rust using a LoRA-tuned Qwen2.5-Coder-7B model with online GRPO reinforcement learning. The reward signal comes from `cargo check` compilation results.

**Runtime**: GPU (T4 or better) — set via *Runtime → Change runtime type*

## 1. Install Dependencies

In [ ]:
!pip install -q -U \
    torch \
    transformers>=4.40.0 \
    accelerate>=0.28.0 \
    peft>=0.10.0 \
    trl>=0.12.0 \
    datasets>=2.18.0 \
    bitsandbytes>=0.41.0 \
    libclang>=16.0.0

In [ ]:
# Install Rust + Cargo (needed for the reward function)
import subprocess, os

result = subprocess.run(
    'curl --proto "=https" --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y',
    shell=True, capture_output=True, text=True
)
print(result.stdout[-2000:] if result.stdout else result.stderr[-2000:])

# Add cargo to PATH for this session
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

# Verify
!cargo --version

## 2. (Optional) Mount Google Drive for Checkpoints

In [ ]:
# Uncomment to mount Drive and save LoRA adapters there
# from google.colab import drive
# drive.mount('/content/drive')
# ADAPTER_DIR = "/content/drive/MyDrive/qwen_lora_adapters"

ADAPTER_DIR = "/content/lora_adapters"  # local (lost on session end)

## 3. Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
C_SOURCE_DIR   = "/content/c_source"       # upload your C repo here
RUST_OUTPUT    = "/content/rust_output"
MIGRATOR_DATA  = "/content/migrator_data"
STATE_FILE     = "/content/migration_state.json"

# ── Model ──────────────────────────────────────────────────────────────────
MODEL_NAME     = "Qwen/Qwen2.5-Coder-7B-Instruct"
HF_TOKEN       = ""   # set if the model requires authentication

# ── Training ───────────────────────────────────────────────────────────────
ONLINE_TRAINING = True   # False = inference only, no GRPO updates
GROUP_SIZE      = 4      # parallel samples per module
LEARNING_RATE   = 5e-6
MAX_NEW_TOKENS  = 1024
LORA_R          = 16
LORA_ALPHA      = 32
SAVE_EVERY      = 10     # save LoRA adapters every N GRPO steps

# ── Debug ──────────────────────────────────────────────────────────────────
LOG_LEVEL = "INFO"   # INFO | DEBUG | NONE

## 4. Upload Your C Source

Either upload a zip and extract it, or use the file browser on the left to upload files into `/content/c_source/`.

In [ ]:
import os
os.makedirs(C_SOURCE_DIR, exist_ok=True)

# --- Option A: upload a zip ---
# from google.colab import files
# uploaded = files.upload()   # pick your zip
# zip_name = list(uploaded.keys())[0]
# !unzip -q "{zip_name}" -d "{C_SOURCE_DIR}"

# --- Option B: clone a git repo ---
# !git clone https://github.com/your/c-repo "{C_SOURCE_DIR}"

print("C source files found:")
c_files = []
for r, _, fs in os.walk(C_SOURCE_DIR):
    for f in fs:
        if f.endswith((".c", ".h")):
            c_files.append(os.path.join(r, f))
print(f"  {len(c_files)} .c/.h files")

## 5. Module Definitions

### 5a. Utility Functions (from C2RustAI)

In [ ]:
import re

def strip_self_includes(c_code: str, module_name: str) -> str:
    pattern = rf'#include\s+"{re.escape(module_name)}(\.h|\.c)"'
    return re.sub(pattern, "", c_code)


def clean_rust_output(text: str) -> str:
    text = re.sub(r"```[a-zA-Z]*", "", text).replace("```", "")
    lines = text.splitlines()
    for i, line in enumerate(lines):
        if line.strip().startswith(("pub ", "fn ", "use ", "mod ", "#[")):
            return "\n".join(lines[i:]).strip()
    return text.strip()


def remove_self_import(rust_code: str, module_name: str) -> str:
    return "\n".join(
        line for line in rust_code.splitlines()
        if f"use crate::{module_name}" not in line
    )

print("Utility functions defined.")

### 5b. Reward Function

In [ ]:
import os, re, subprocess, tempfile
from dataclasses import dataclass, field

_CARGO_TOML = """\
[package]
name = "reward_check"
version = "0.1.0"
edition = "2021"

[dependencies]
"""

_ERROR_PENALTY  = 0.12
_WARN_PENALTY   = 0.05
_UNSAFE_PENALTY = 0.10
_UNWRAP_PENALTY = 0.02


@dataclass
class RewardInfo:
    compilation_score: float = 0.0
    error_count:       int   = 0
    warning_count:     int   = 0
    unsafe_count:      int   = 0
    unwrap_count:      int   = 0
    errors:            list  = field(default_factory=list)
    total:             float = 0.0


def _write_project(tmpdir: str, rust_code: str) -> None:
    src_dir = os.path.join(tmpdir, "src")
    os.makedirs(src_dir, exist_ok=True)
    with open(os.path.join(tmpdir, "Cargo.toml"), "w") as f:
        f.write(_CARGO_TOML)
    deps = set(re.findall(r"use\s+crate::(\w+)", rust_code))
    mod_header = "\n".join(f"pub mod {d};" for d in deps)
    entry = "main.rs" if re.search(r"\bfn\s+main\s*\(", rust_code) else "lib.rs"
    full_code = (mod_header + "\n\n" if mod_header else "") + rust_code
    with open(os.path.join(src_dir, entry), "w", encoding="utf-8") as f:
        f.write(full_code)
    for dep in deps:
        with open(os.path.join(src_dir, f"{dep}.rs"), "w") as f:
            f.write("// stub\n")


def _run_cargo_check(tmpdir: str, timeout: int = 30):
    try:
        result = subprocess.run(
            ["cargo", "check", "--quiet"],
            cwd=tmpdir, capture_output=True, text=True, timeout=timeout,
        )
        return result.returncode == 0, result.stderr
    except subprocess.TimeoutExpired:
        return None, "timeout"
    except FileNotFoundError:
        return None, "cargo_not_found"


def _parse_diagnostics(stderr: str):
    errors, warning_count = [], 0
    for line in stderr.splitlines():
        s = line.lstrip()
        if s.startswith("error[") or (
            s.startswith("error") and "aborting" not in s and "could not compile" not in s
        ):
            errors.append(s)
        elif s.startswith("warning[") or (s.startswith("warning") and "warning(s)" not in s):
            warning_count += 1
    return len(errors), warning_count, errors


def compute_reward(rust_code: str, module_name: str = "module", timeout: int = 30):
    info = RewardInfo()
    if not rust_code.strip():
        return 0.0, info
    info.unsafe_count = len(re.findall(r"\bunsafe\s*\{", rust_code))
    info.unwrap_count = len(re.findall(r"\.(unwrap|expect)\s*\(", rust_code))
    with tempfile.TemporaryDirectory() as tmpdir:
        _write_project(tmpdir, rust_code)
        success, stderr = _run_cargo_check(tmpdir, timeout)
    if success is None:
        info.compilation_score = 0.5
    elif success:
        info.compilation_score = 1.0
        _, info.warning_count, _ = _parse_diagnostics(stderr)
    else:
        info.error_count, info.warning_count, info.errors = _parse_diagnostics(stderr)
        info.compilation_score = max(0.0, 1.0 - info.error_count * _ERROR_PENALTY)
    raw = (
        info.compilation_score
        - _WARN_PENALTY   * min(info.warning_count, 5)
        - _UNSAFE_PENALTY * min(info.unsafe_count,  3)
        - _UNWRAP_PENALTY * min(info.unwrap_count,  5)
    )
    info.total = max(0.0, min(1.0, raw))
    return info.total, info

print("Reward function defined.")

### 5c. C Repository Analyzer

In [ ]:
import os, json
import clang.cindex
from clang.cindex import CursorKind, TypeKind, TranslationUnit


class CToRustAnalyzer:
    def __init__(self, repo_path):
        self.repo_path = os.path.abspath(repo_path)
        self.index = clang.cindex.Index.create()
        self.ast_data = {}
        self.pointer_data = []
        self.dep_graph = {"modules": {}, "call_graph": {}}

    def _get_rel_path(self, path):
        return os.path.relpath(path, self.repo_path)

    def _get_module_name(self, filepath):
        return os.path.splitext(os.path.basename(filepath))[0]

    def analyze_repository(self):
        for root, _, files in os.walk(self.repo_path):
            for file in files:
                if file.endswith((".c", ".h")):
                    self._process_file(os.path.join(root, file))

    def _process_file(self, filepath):
        rel_path    = self._get_rel_path(filepath)
        folder      = os.path.dirname(rel_path) or "root"
        module_name = self._get_module_name(filepath)
        print(f"Analyzing {rel_path}...")
        options = TranslationUnit.PARSE_DETAILED_PROCESSING_RECORD
        args    = ["-x", "c", f"-I{self.repo_path}"]
        try:
            tu = self.index.parse(filepath, args=args, options=options)
            if folder not in self.dep_graph["modules"]:
                self.dep_graph["modules"][folder] = {}
            if module_name not in self.dep_graph["modules"][folder]:
                self.dep_graph["modules"][folder][module_name] = {
                    "metrics": {"import_count": 0, "export_count": 0},
                    "includes": [], "exports": []
                }
            file_entry = self.dep_graph["modules"][folder][module_name]
            self.ast_data[os.path.basename(filepath)] = self._serialize_ast(tu.cursor, filepath)
            self._walk_semantics(tu.cursor, filepath, file_entry, current_func=None)
            file_entry["includes"] = list(set(file_entry["includes"]))
            file_entry["exports"]  = list(set(file_entry["exports"]))
            file_entry["metrics"]["import_count"] = len(file_entry["includes"])
            file_entry["metrics"]["export_count"] = len(file_entry["exports"])
        except Exception as e:
            print(f"Failed to parse {rel_path}: {e}")

    def _serialize_ast(self, cursor, origin_file):
        if cursor.kind != CursorKind.TRANSLATION_UNIT:
            if not cursor.location.file or \
               os.path.abspath(cursor.location.file.name) != os.path.abspath(origin_file):
                return None
        node = {
            "kind": str(cursor.kind).split(".")[1],
            "name": cursor.spelling,
            "type": cursor.type.spelling or None,
            "location": f"{cursor.location.line}:{cursor.location.column}" if cursor.location.file else "0:0",
            "children": []
        }
        for child in cursor.get_children():
            s = self._serialize_ast(child, origin_file)
            if s:
                node["children"].append(s)
        return node

    def _walk_semantics(self, cursor, filepath, file_entry, current_func):
        is_in_file = (cursor.location.file and
                      os.path.abspath(cursor.location.file.name) == os.path.abspath(filepath))
        if cursor.kind == CursorKind.INCLUSION_DIRECTIVE:
            file_entry["includes"].append(cursor.spelling)
        if cursor.kind == CursorKind.FUNCTION_DECL and is_in_file and cursor.is_definition():
            current_func = cursor.spelling
            if current_func not in self.dep_graph["call_graph"]:
                self.dep_graph["call_graph"][current_func] = set()
            file_entry["exports"].append(current_func)
        if cursor.kind == CursorKind.CALL_EXPR and current_func and cursor.spelling:
            self.dep_graph["call_graph"][current_func].add(cursor.spelling)
        if is_in_file and cursor.type.kind in (TypeKind.POINTER, TypeKind.INCOMPLETEARRAY) and \
           cursor.kind in (CursorKind.VAR_DECL, CursorKind.PARM_DECL, CursorKind.FIELD_DECL):
            self.pointer_data.append({
                "file": os.path.basename(filepath), "name": cursor.spelling,
                "parent_func": current_func, "type": cursor.type.spelling,
                "is_const": cursor.type.get_pointee().is_const_qualified()
            })
        for child in cursor.get_children():
            self._walk_semantics(child, filepath, file_entry, current_func)

    def _finalize_call_graph(self):
        for func in self.dep_graph["call_graph"]:
            self.dep_graph["call_graph"][func] = sorted(list(self.dep_graph["call_graph"][func]))

    def export_json(self, output_dir):
        os.makedirs(output_dir, exist_ok=True)
        self._finalize_call_graph()
        for name, data in {
            "ast.json": self.ast_data,
            "pointers.json": self.pointer_data,
            "dependencies.json": self.dep_graph,
            "call_graph.json": self.dep_graph["call_graph"],
        }.items():
            with open(os.path.join(output_dir, name), "w") as f:
                json.dump(data, f, indent=2)
        print(f"Exported JSON files to {output_dir}/")

print("Analyzer defined.")

### 5d. Migration State Helpers

In [ ]:
import json, os

# ── memory_generate ────────────────────────────────────────────────────────

def _normalize_import(name):
    base = os.path.basename(name)
    return os.path.splitext(base)[0] if base.endswith((".h", ".c")) else base


def initialize_migration_state(deps_path, state_path="migration_state.json"):
    if os.path.exists(state_path):
        print(f"State file already exists at {state_path}. Skipping.")
        return
    with open(deps_path) as f:
        deps = json.load(f)
    state = {"modules": {}, "global_stats": {"total_files": 0, "migrated_files": 0}}
    repo_modules = {
        mn for fe in deps.get("modules", {}).values() for mn in fe.keys()
    }
    for folder, module_entries in deps.get("modules", {}).items():
        state["modules"][folder] = {}
        for module_name, data in module_entries.items():
            state["global_stats"]["total_files"] += 1
            filtered_imports = {
                inc: False for inc in data.get("includes", [])
                if _normalize_import(inc) in repo_modules
                and _normalize_import(inc) != module_name
            }
            state["modules"][folder][module_name] = {
                "migrated": False,
                "exports":  {n: False for n in data.get("exports", [])},
                "imports":  filtered_imports,
                "stats": {
                    "total_imports":    len(filtered_imports),
                    "total_exports":    len(data.get("exports", [])),
                    "migrated_imports": 0,
                    "migrated_exports": 0,
                },
            }
    with open(state_path, "w") as f:
        json.dump(state, f, indent=2)
    print(f"Initialized migration state at {state_path}")


# ── update_memory ──────────────────────────────────────────────────────────

def reconcile_migration_progress(state_path):
    with open(state_path) as f:
        state = json.load(f)
    fully_migrated = {
        mn for folder in state["modules"].values()
        for mn, md in folder.items()
        if md["migrated"] or (md["exports"] and all(md["exports"].values()))
    }
    changes, total_migrated = 0, 0
    g_ti = g_te = g_mi = g_me = 0
    for folder in state["modules"].values():
        for module_name, md in folder.items():
            for imp in list(md["imports"]):
                if _normalize_import(imp) in fully_migrated and not md["imports"][imp]:
                    md["imports"][imp] = True
                    changes += 1
            all_imp = all(md["imports"].values()) if md["imports"] else True
            all_exp = all(md["exports"].values()) if md["exports"] else True
            if all_imp and all_exp and not md["migrated"]:
                md["migrated"] = True
                changes += 1
            if md["migrated"]:
                total_migrated += 1
            mi = sum(1 for v in md["imports"].values() if v)
            me = sum(1 for v in md["exports"].values() if v)
            md["stats"] = {
                "total_imports":    len(md["imports"]),
                "total_exports":    len(md["exports"]),
                "migrated_imports": mi,
                "migrated_exports": me,
            }
            g_ti += len(md["imports"]); g_te += len(md["exports"])
            g_mi += mi;                 g_me += me
    state["global_stats"].update(
        migrated_files=total_migrated,
        total_imports=g_ti, total_exports=g_te,
        migrated_imports=g_mi, migrated_exports=g_me,
    )
    if changes:
        with open(state_path, "w") as f:
            json.dump(state, f, indent=2)
        print(f"Reconciliation: {changes} updates saved.")
    else:
        print("No new progress detected.")


# ── mark_migrated ──────────────────────────────────────────────────────────

def mark_node_migrated(json_path: str, node_path: str) -> None:
    with open(json_path) as f:
        data = json.load(f)
    node = data["modules"]
    for part in node_path.split("/"):
        node = node[part]
    node["migrated"] = True
    if "exports" in node:
        node["exports"] = {k: True for k in node["exports"]}
    if "stats" in node:
        node["stats"]["migrated_exports"] = node["stats"].get("total_exports", 0)
    with open(json_path, "w") as f:
        json.dump(data, f, indent=2)


# ── choose_module ──────────────────────────────────────────────────────────

import heapq
from collections import defaultdict

def _normalize_dep(path: str) -> str:
    return path.replace(".h", "").replace(".c", "").replace("../", "").replace("./", "")


def _build_dep_graph(data):
    modules, deps, dependents = {}, defaultdict(set), defaultdict(set)
    for group, subs in data["modules"].items():
        for name in subs:
            modules[f"{group}/{name}"] = subs[name]
    for group, subs in data["modules"].items():
        for name, info in subs.items():
            src = f"{group}/{name}"
            for imp in info.get("imports", {}):
                dep_key = _normalize_dep(imp)
                matched = next((n for n in modules if n.split("/")[-1] == dep_key), None)
                if matched:
                    deps[src].add(matched)
                    dependents[matched].add(src)
    return modules, deps, dependents


def pick_next_module(json_path: str) -> str | None:
    with open(json_path) as f:
        data = json.load(f)
    nodes, deps, dependents = _build_dep_graph(data)
    migrated = {
        f"{g}/{n}" for g, subs in data["modules"].items()
        for n, info in subs.items() if info.get("migrated", False)
    }
    heap = []
    for node, info in nodes.items():
        if node in migrated:
            continue
        if not [d for d in deps[node] if d not in migrated]:
            score = (len(info.get("exports", {})), len(deps[node]), len(dependents[node]))
            heapq.heappush(heap, (score, node))
    return heapq.heappop(heap)[1] if heap else None


# ── gen_cargo ──────────────────────────────────────────────────────────────

def generate_cargo(root: str, package_name: str) -> None:
    print("\nGenerating Cargo project...")
    src_dir = os.path.join(root, "src")
    os.makedirs(src_dir, exist_ok=True)
    for f in [x for x in os.listdir(root) if x.endswith(".rs")]:
        os.rename(os.path.join(root, f), os.path.join(src_dir, f))
    all_rs = [f for f in os.listdir(src_dir) if f.endswith(".rs")]
    if not all_rs:
        print("WARNING: No .rs files found.")
        return
    entry = next(
        (f for f in all_rs if re.search(r"\bfn\s+main\s*\(", open(os.path.join(src_dir, f)).read())),
        all_rs[0]
    )
    main_rs = os.path.join(src_dir, "main.rs")
    if entry != "main.rs":
        os.rename(os.path.join(src_dir, entry), main_rs)
    modules = [os.path.splitext(f)[0] for f in os.listdir(src_dir)
               if f.endswith(".rs") and f != "main.rs"]
    mod_header = "\n".join(f"mod {m};" for m in modules)
    with open(main_rs) as f:
        code = f.read()
    with open(main_rs, "w") as f:
        f.write(f"{mod_header}\n\n{code}" if mod_header else code)
    toml = f'[package]\nname = "{package_name}"\nversion = "0.1.0"\nedition = "2021"\n\n[dependencies]\n'
    with open(os.path.join(root, "Cargo.toml"), "w") as f:
        f.write(toml)
    print(f"Cargo project ready: {root}")

print("Migration state helpers defined.")

### 5e. Qwen2.5 Model + GRPO Training Engine

In [ ]:
import os, re, time
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


def _log(msg: str):
    if LOG_LEVEL == "NONE":
        return
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

_ADV_EPS = 1e-8

_state = {
    "tokenizer": None, "model": None, "optimizer": None, "step": 0
}

_SYSTEM_PROMPT = (
    "You are a compiler-like C-to-Rust translator. "
    "Output ONLY valid Rust source code. "
    "No explanations, no markdown fences, no commentary."
)


def _build_messages(c_code: str, include_modules: list) -> list:
    dep_hint = (
        "\n".join(f"- {m} → use crate::{m}::*;" for m in include_modules)
        if include_modules else "(none)"
    )
    return [
        {"role": "system", "content": _SYSTEM_PROMPT},
        {"role": "user", "content": (
            "Convert the following C code to Rust.\n\n"
            f"Dependencies:\n{dep_hint}\n\n"
            f"```c\n{c_code}\n```"
        )},
    ]


def _ensure_model():
    if _state["model"] is not None:
        return _state["tokenizer"], _state["model"], _state["optimizer"]

    _log(f"Loading tokenizer: {MODEL_NAME}")
    login_kwargs = {"token": HF_TOKEN} if HF_TOKEN else {}
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, **login_kwargs)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    _log("Loading base model (QLoRA 4-bit)...")
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        **login_kwargs,
    )

    if ONLINE_TRAINING:
        base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
        _log("Attaching LoRA adapters...")
        lora_cfg = LoraConfig(
            r=LORA_R, lora_alpha=LORA_ALPHA,
            target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
            lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        )
        model = get_peft_model(base, lora_cfg)
        model.print_trainable_parameters()
    else:
        model = base

    model.eval()
    optimizer = None
    if ONLINE_TRAINING:
        trainable = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(trainable, lr=LEARNING_RATE)
        _log(f"Optimizer ready | {len(trainable)} trainable param tensors")

    _state.update(tokenizer=tokenizer, model=model, optimizer=optimizer)
    return tokenizer, model, optimizer


def _generate(tokenizer, model, prompt: str, n: int):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    _log(f"Generating {n} samples...")
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True, temperature=0.8, top_p=0.9,
            pad_token_id=tokenizer.eos_token_id, num_return_sequences=n,
        )
    texts, resp_ids = [], []
    for i in range(n):
        ids  = out[i][prompt_len:]
        text = tokenizer.decode(ids, skip_special_tokens=True)
        texts.append(text)
        resp_ids.append(ids)
    _log("Generation complete.")
    return texts, resp_ids, inputs


def _grpo_step(model, optimizer, inputs, resp_ids, rewards):
    rewards_t  = torch.tensor(rewards, dtype=torch.float32)
    mean_r     = rewards_t.mean()
    std_r      = rewards_t.std(unbiased=False) + _ADV_EPS
    advantages = (rewards_t - mean_r) / std_r
    _log(f"GRPO | mean={mean_r:.4f} std={std_r:.4f}")
    prompt_len = inputs["input_ids"].shape[1]
    loss_terms = []
    model.train()
    for i, (ids, adv) in enumerate(zip(resp_ids, advantages)):
        full   = torch.cat([inputs["input_ids"][0], ids]).unsqueeze(0)
        labels = full.clone()
        labels[:, :prompt_len] = -100
        out    = model(full, labels=labels)
        loss_terms.append(adv.to(model.device) * out.loss)
    if loss_terms:
        loss = torch.stack(loss_terms).mean()
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        _log(f"GRPO update | loss={loss.item():.4f}")
        return loss.item()
    return 0.0


def convert_c_to_rust(file_path: str, output_path: str) -> str:
    module_name = os.path.splitext(os.path.basename(file_path))[0]
    _log(f"Converting: {module_name}")
    with open(file_path) as f:
        c_code = f.read()
    c_code = strip_self_includes(c_code, module_name)
    tokenizer, model, optimizer = _ensure_model()
    includes = [
        os.path.splitext(os.path.basename(inc))[0]
        for inc in re.findall(r'#include\s+"([^"]+)"', c_code)
        if os.path.splitext(os.path.basename(inc))[0] != module_name
    ]
    prompt = tokenizer.apply_chat_template(
        _build_messages(c_code, includes),
        tokenize=False, add_generation_prompt=True,
    )
    texts, resp_ids, inputs = _generate(tokenizer, model, prompt, GROUP_SIZE)
    rewards = [compute_reward(clean_rust_output(t), module_name)[0] for t in texts]
    _log("Rewards: " + " | ".join(f"{r:.4f}" for r in rewards))
    best = max(range(len(rewards)), key=lambda i: rewards[i])
    rust = remove_self_import(clean_rust_output(texts[best]), module_name)
    if ONLINE_TRAINING:
        _state["step"] += 1
        loss = _grpo_step(model, optimizer, inputs, resp_ids, rewards)
        _log(f"Step {_state['step']} | loss={loss:.4f}")
        if _state["step"] % SAVE_EVERY == 0 and hasattr(model, "save_pretrained"):
            model.save_pretrained(ADAPTER_DIR)
            _log(f"Adapters saved → {ADAPTER_DIR}")
    os.makedirs(output_path, exist_ok=True)
    out_file = os.path.join(output_path, f"{module_name}.rs")
    with open(out_file, "w", encoding="utf-8") as f:
        f.write(rust)
    _log(f"Saved → {out_file}")
    return f"{module_name}.rs"

print("Model + training engine defined.")

## 6. Analyze C Repository

In [ ]:
import os, json

deps_path = os.path.join(MIGRATOR_DATA, "dependencies.json")

if not os.path.exists(deps_path):
    print("Running C repository analysis...")
    analyzer = CToRustAnalyzer(C_SOURCE_DIR)
    analyzer.analyze_repository()
    analyzer.export_json(MIGRATOR_DATA)
else:
    print(f"Analysis already exists at {MIGRATOR_DATA}/")

# Initialize migration state
initialize_migration_state(deps_path, STATE_FILE)

with open(STATE_FILE) as f:
    s = json.load(f)
gs = s["global_stats"]
print(f"\nModules to migrate: {gs['total_files']}")

## 7. Run Migration + GRPO Training Loop

In [ ]:
import json, os, time

os.makedirs(RUST_OUTPUT, exist_ok=True)

def _all_migrated():
    with open(STATE_FILE) as f:
        data = json.load(f)
    gs = data["global_stats"]
    return gs["migrated_files"] >= gs["total_files"]


step = 0
while not _all_migrated():
    node = pick_next_module(STATE_FILE)
    if node is None:
        print("No valid next module — possible dependency cycle. Stopping.")
        break

    node_clean = node.removeprefix("root/")
    if not node_clean.endswith(".c"):
        node_clean += ".c"

    c_file = os.path.join(C_SOURCE_DIR, node_clean)
    if not os.path.exists(c_file):
        print(f"  Skipping (not found): {c_file}")
        mark_node_migrated(STATE_FILE, node)
        reconcile_migration_progress(STATE_FILE)
        continue

    step += 1
    print(f"\n[{step}] Migrating: {node}")
    rust_file = convert_c_to_rust(c_file, RUST_OUTPUT)
    print(f"  → {rust_file}")

    mark_node_migrated(STATE_FILE, node)
    reconcile_migration_progress(STATE_FILE)

    with open(STATE_FILE) as f:
        gs = json.load(f)["global_stats"]
    print(f"  Progress: {gs['migrated_files']}/{gs['total_files']} modules")
    time.sleep(0.3)

print("\nMigration complete!")

## 8. Package as Cargo Project

In [ ]:
import re

raw_name  = os.path.basename(os.path.abspath(C_SOURCE_DIR))
pkg_name  = re.sub(r"[^a-zA-Z0-9_]", "_", raw_name)
if pkg_name and pkg_name[0].isdigit():
    pkg_name = "_" + pkg_name

generate_cargo(RUST_OUTPUT, pkg_name)

print("\nFinal Cargo project layout:")
for root, dirs, files in os.walk(RUST_OUTPUT):
    level = root.replace(RUST_OUTPUT, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")

## 9. Save Final LoRA Adapters

In [ ]:
if _state["model"] is not None and ONLINE_TRAINING and hasattr(_state["model"], "save_pretrained"):
    _state["model"].save_pretrained(ADAPTER_DIR)
    _state["tokenizer"].save_pretrained(ADAPTER_DIR)
    print(f"Final adapters saved to: {ADAPTER_DIR}")
    print(f"Total GRPO steps: {_state['step']}")
else:
    print("No training was performed (ONLINE_TRAINING=False or model not loaded).")

## 10. (Optional) Download Adapters

In [ ]:
# Download the LoRA adapters as a zip
# import shutil
# from google.colab import files
# shutil.make_archive("/content/lora_adapters", "zip", ADAPTER_DIR)
# files.download("/content/lora_adapters.zip")

# Download the Rust output as a zip
# shutil.make_archive("/content/rust_output", "zip", RUST_OUTPUT)
# files.download("/content/rust_output.zip")
print("Uncomment lines above to download outputs.")